# Project 07 · Content Pipeline Graph (Starter notebook)

**Module 07 · LangGraph Fundamentals** · Part 1 (Core)

| | |
|---|---|
| ⏱️ Estimated time | 5–7 hours |
| 📋 Prerequisites | Lessons 07.01–07.07 |
| 🔑 Keys | `GROQ_API_KEY` (LIVE mode, optional) · `LANGSMITH_API_KEY` (traces, optional) |
| 💸 Cost (LIVE) | $0 on the free tier: LIVE runs the pipeline 5–6 times (first build, 3 topics, the server run, acceptance check 1), roughly 35–70k tokens of `gpt-oss-20b` plus the `gpt-oss-120b` critic (well under $0.05 at paid prices). `LIVE_TOPICS = 1` cuts the topic runs only |
| 🧠 Kernel | **Python (lc-mastery)** |
| ✅ Docs verified | 2026-10-12 |
| 📄 Spec | [project/README.md](README.md) (also in [COURSE_OUTLINE.md](../../../COURSE_OUTLINE.md#project-07--content-pipeline-graph)) |

**Goal:** a graph that turns a topic into a reviewed blog-post draft: plan → parallel research → draft → critique ⟲ revise → finalize. The same graph runs here, under `langgraph dev`, and in Studio.

This is the **starter**. Every `# TODO` marks work for you. The notebook runs top to bottom even before you finish: unfinished parts show ❌ in the checks instead of crashing. Stuck? Peek at the [reference solution](solutions/07-project-content-pipeline-solution.ipynb).

## Requirements (from the spec)

1. Nodes `plan_outline` → parallel `research_section` (**`Send`**) → `write_draft` → `critique` → conditional loop to `revise` (evaluator-optimizer, **max 3 iterations**) → `finalize`.
2. **Typed state with reducers** for the collected sections; **structured outputs** for the outline and the critique.
3. Graph image in the notebook; the app runs under **`langgraph dev`** and is inspected in **Studio**.
4. **Cost/latency per run** logged.

**Acceptance criteria:** deterministic termination · the same app runs in the notebook and in Studio · the critique score improves (or the loop stops) within 3 iterations on 3 test topics.

**Modes:** with `GROQ_API_KEY` the notebook is **LIVE** (`gpt-oss-20b` plans, writes and critiques). Without it, **OFFLINE DEMO**: scripted models from `mastery.graphs.content.offline_models()`. The scripted critic scores 4 + 2 per fixed issue and the scripted writer fixes one issue per revision, so the loop *visibly* improves (4 → 6 → 8) and the checks test **your graph**, not a model. Research always comes from the Module 06 corpus (BM25, offline).

| TODO | Where | What |
|---|---|---|
| 1 | Step 2 | `fan_out`: one `Send("research_section", ...)` per outline section |
| 2 | Step 2 | `revise`: rewrite the draft from the critique's issues, count the iteration |
| 3 | Step 2 | `after_critique`: `"finalize"` at score ≥ threshold **or** after `MAX_ITERATIONS`, else `"revise"` |
| 4 | Step 5 | `cost_latency_table`: seconds per node from `run_log` + tokens and $ from usage |

## Step 0 · Setup

In [ ]:
%pip install --quiet "langgraph>=1.2.14,<2" "langchain-core>=1.6.9,<2" "langchain-groq>=1.1.3,<2" "langgraph-cli[inmem]>=0.4.33,<1" "langgraph-api>=0.16.0,<1"

In [ ]:
# Setup 1: Find the repository root (Jupyter starts in modules/07-langgraph-fundamentals/).
import importlib.util
import os
import sys
from pathlib import Path

# Optional: set LC_MASTERY_ROOT to your clone if Jupyter starts outside it (no paths are hardcoded).
_override = os.environ.get("LC_MASTERY_ROOT", "").strip()
REPO_ROOT = next(
    (p for p in ([Path(_override).expanduser()] if _override else [Path.cwd(), *Path.cwd().parents])
     if (p / "pyproject.toml").is_file() and (p / "src" / "mastery").is_dir()),
    None,
)
if REPO_ROOT is None:
    raise FileNotFoundError("Could not find the course repo. Start Jupyter inside your clone (or set LC_MASTERY_ROOT to it).")

# Setup 2: Import `mastery` from THIS clone's src/, like pytest does (pythonpath = ["src"]), and check
# that the env's own install (00.02 guide, Step 5) points here too: terminal commands use that one.
SRC_DIR = (REPO_ROOT / "src").resolve()
installed_spec = importlib.util.find_spec("mastery")
if sys.path[0] != str(SRC_DIR):
    sys.path.insert(0, str(SRC_DIR))
import mastery

print("Repo root:", REPO_ROOT)
print("mastery  :", mastery.__version__, "from", mastery.__file__)
installed_at = installed_spec.origin if installed_spec else None
if not Path(mastery.__file__).resolve().is_relative_to(SRC_DIR):
    print("⚠️ This kernel had already imported mastery from another folder: restart the kernel, run from the top.")
elif installed_at is None or not Path(installed_at).resolve().is_relative_to(SRC_DIR):
    print("⚠️ This env's own mastery install does not use this clone's src/:", installed_at or "(not installed)")
    print("   This notebook uses this clone's src/ anyway, but terminal commands (python -m mastery...) do not.")
    print('   Fix once (Git Bash, course env active, root of THIS clone): python -m pip install -e ".[dev]"')

In [ ]:
# Setup 4: Silence known, harmless noise so real problems stand out.
import os
import warnings
from importlib.metadata import version

os.environ.setdefault("ANONYMIZED_TELEMETRY", "False")          # Chroma (07.07 reuses Module 06): no usage telemetry.
# Same as Lesson 02.04: older langchain-core 1.6.x Pydantic serializer warning about the injected runtime.
# Only older releases raise it; on current ones this filter is skipped.
if tuple(int("".join(c for c in x if c.isdigit()) or 0) for x in version("langchain-core").split(".")[:3]) < (1, 6, 11):
    warnings.filterwarnings("ignore", message="Pydantic serializer warnings", category=UserWarning)

In [ ]:
# Imports and settings used by the whole notebook.
import inspect
import json
import os
import re
import time

from langchain_core.callbacks import get_usage_metadata_callback
from langgraph.graph import END, START, StateGraph
from langgraph.types import Send

from mastery.config import is_set, load_env
from mastery.graphs import ContentState, Critique, Outline, SectionNote, run_summary, show_graph, timed
from mastery.graphs.content import (TEST_TOPICS, content_recursion_limit, corpus_researcher, critic_prompt,
                                    offline_models, planner_prompt, writer_prompt)
from mastery.pricing import estimate_cost

load_env()
LIVE = is_set("GROQ_API_KEY")
os.environ.setdefault("LANGSMITH_PROJECT", "m07-project-content-pipeline")   # CONVENTIONS: mNN-<slug>, unless .env sets one
LIVE_TOPICS = None           # LIVE only: e.g. 1 to run just the first topic (saves tokens). None = all 3.
RUN_SERVER = True            # Step 6: start `langgraph dev` for the app (local, free; ~5-20 s).
RUN_STRETCH = True           # Step 8: Functional API version (offline models, free).
MAX_ITERATIONS, THRESHOLD = 3, 8
OUT = REPO_ROOT / "data" / "outputs" / "m07"
OUT.mkdir(parents=True, exist_ok=True)
APP_DIR = REPO_ROOT / "apps" / "m07_content_pipeline"

if LIVE:
    from mastery.models import get_model

    llm = get_model("fast", temperature=0, reasoning_effort="low")
    # The judge is not the writer: a model grading its own drafts is too kind (Lesson 07.04).
    MODELS = {"planner": llm, "writer": llm, "critic": get_model("reasoning", temperature=0, reasoning_effort="low")}
    MODEL_KEY = "groq:" + llm.model_name
else:
    MODELS = offline_models()
    MODEL_KEY = "groq:openai/gpt-oss-20b"     # for the $ column (offline usage is 0 tokens)
researcher = corpus_researcher()
TOPICS = TEST_TOPICS[:LIVE_TOPICS] if LIVE and LIVE_TOPICS else TEST_TOPICS


def check(label: str, ok: bool | None, detail: str = "") -> bool | None:
    """ok=None means the check was skipped (printed ⏭️ and counted apart, never as a pass)."""
    print("⏭️" if ok is None else "✅" if ok else "❌", label, f"({detail})" if detail and ok is False else "")
    return None if ok is None else bool(ok)


print("Mode  :", "LIVE (gpt-oss-20b)" if LIVE else "OFFLINE DEMO (scripted models, no key, free)")
print("Topics:", TOPICS)

**You should now see** the mode (OFFLINE DEMO without a key) and the three test topics.

## Step 1 · State and structured outputs (provided)

Read the state before you write nodes: `sections`, `scores` and `run_log` have an `operator.add` **reducer**, so three parallel `research_section` workers each *add* a note instead of overwriting each other (Lesson 07.02). `outline` and `critique` hold Pydantic objects from **structured output** (Lesson 03.x / 07.04).

In [ ]:
# The state (provided, from mastery.graphs.state): reducers make parallel results add up.
print(inspect.getsource(ContentState))
print(Outline.model_json_schema()["required"], "|", Critique.model_json_schema()["required"])

**You should now see** the `ContentState` class with its `Annotated[..., operator.add]` fields, then `['title', 'sections'] | ['score']` (the required fields; `issues` defaults to an empty list).

## Step 2 · Build the graph

**TODO 1–3** are in the second cell below.

In [ ]:
# Provided nodes. Each is wrapped in @timed: it adds {"node", "seconds"} to run_log (Lesson 07.06).
plan_model = MODELS["planner"].with_structured_output(Outline)      # structured output #1
critic_model = MODELS["critic"].with_structured_output(Critique)    # structured output #2
writer = MODELS["writer"]


@timed("plan_outline")
def plan_outline(state: ContentState) -> dict:
    return {"outline": plan_model.invoke(planner_prompt(state["topic"])), "iteration": 0}


@timed("research_section")
def research_section(task: dict) -> dict:
    notes, sources = researcher(task["question"])
    return {"sections": [SectionNote(title=task["title"], notes=notes, sources=sources)]}


@timed("write_draft")
def write_draft(state: ContentState) -> dict:
    return {"draft": writer.invoke(writer_prompt(state["topic"], state["outline"], state["sections"])).text}


@timed("critique")
def critique(state: ContentState) -> dict:
    review = critic_model.invoke(critic_prompt(state["topic"], state["draft"]))
    return {"critique": review, "scores": [review.score]}


@timed("finalize")
def finalize(state: ContentState) -> dict:
    sources = sorted({src for s in state["sections"] for src in s["sources"]})
    return {"final": state["draft"].rstrip() + "\n\n---\nSources: " + (", ".join(sources) or "none")}

In [ ]:
# Your three pieces of the graph.
def fan_out(state: ContentState) -> list[Send]:
    """One research task per outline section: Send("research_section", {"title": ..., "question": ...})."""
    # TODO 1: fan out one research task per outline section; each worker needs that section's title and question
    #         (Lesson 07.03 Part 2).
    return []


@timed("revise")
def revise(state: ContentState) -> dict:
    """Rewrite the draft using the critique's issues; count the iteration."""
    # TODO 2: rewrite the draft so it addresses the critique's issues (writer_prompt takes them), and count the revision.
    return {"iteration": state.get("iteration", 0) + 1}


def after_critique(state: ContentState) -> str:
    """"finalize" when the score reaches THRESHOLD or after MAX_ITERATIONS revisions; otherwise "revise"."""
    # TODO 3: read state["critique"].score and state.get("iteration", 0). The counter must end the loop on its own.
    return "finalize"

<details><summary>Hints for TODO 1–3 (open only if stuck)</summary>

1. A list comprehension of `Send("research_section", {"title": s.title, "question": s.question})` over `state["outline"].sections`.
2. `writer.invoke(writer_prompt(state["topic"], state["outline"], state["sections"], state["draft"], state["critique"].issues))`; return its `.text` as `draft` and `iteration + 1`.
3. Two exits to `"finalize"`: the score reached `THRESHOLD`, or `iteration >= MAX_ITERATIONS` (that one never depends on the model).
</details>

In [ ]:
# Wire the graph (provided). Node names must match the app's graph: Steps 6–7 check it.
def build_pipeline(checkpointer=None):
    builder = StateGraph(ContentState)
    builder.add_node("plan_outline", plan_outline)
    builder.add_node("research_section", research_section)
    builder.add_node("write_draft", write_draft)
    builder.add_node("critique", critique)
    builder.add_node("revise", revise)
    builder.add_node("finalize", finalize)
    builder.add_edge(START, "plan_outline")
    builder.add_conditional_edges("plan_outline", fan_out, ["research_section"])
    builder.add_edge("research_section", "write_draft")
    builder.add_edge("write_draft", "critique")
    builder.add_conditional_edges("critique", after_critique, ["revise", "finalize"])
    builder.add_edge("revise", "critique")
    builder.add_edge("finalize", END)
    # 07.03's rule: a tight recursion_limit (5 + 2 per revision, + 1, + a small margin), not the 10,000 default.
    return builder.compile(checkpointer=checkpointer, name="content_pipeline").with_config(
        recursion_limit=content_recursion_limit(MAX_ITERATIONS))


pipeline = build_pipeline()
first = pipeline.invoke({"topic": TOPICS[0]})
print("sections :", [s["title"] for s in first.get("sections", [])])
print("scores   :", first.get("scores", []), "| revisions:", first.get("iteration"))
print()
print(first.get("final", first.get("draft", "(no draft: the graph stopped early, see TODO 1)"))[:700])

**You should now see** three section titles (`The basics`, `Practical tips`, `Troubleshooting` offline), `scores   : [4, 6, 8] | revisions: 2` and the start of the article with `[source: ...]` tags. Offline, sections can repeat the same corpus sentences (the BM25 researcher answers each question on its own and copies sentences; it never invents text). A LIVE writer merges them into prose. Starter before the TODOs: no sections, `scores : []` and the no-draft message (with `fan_out` returning nothing, the graph ends after `plan_outline`).

## Step 3 · Look at it
> 🌐 Draws through mermaid.ink (no key, no tokens). Offline? You get the Mermaid text instead.

In [ ]:
_ = show_graph(pipeline)   # Lesson 07.06 showed how to save it: draw_mermaid_png(output_file_path=...)
print("nodes:", [n for n in pipeline.get_graph().nodes if not n.startswith("__")])

**You should now see** the diagram (`research_section` with one incoming edge from `plan_outline`, the `critique` ⇄ `revise` loop) and the six node names.

## Step 4 · Run the 3 test topics

In [ ]:
# Run the 3 test topics; collect state, latency and token usage per run.
runs = []
for topic in TOPICS:
    start = time.perf_counter()
    with get_usage_metadata_callback() as usage:
        out = pipeline.invoke({"topic": topic})
    runs.append({"topic": topic, "out": out, "seconds": round(time.perf_counter() - start, 3),
                 "usage": dict(usage.usage_metadata)})
    print(f"{topic:<32} scores={out.get('scores', [])} revisions={out.get('iteration')} "
          f"{runs[-1]['seconds']}s final={'final' in out}")

**You should now see** one line per topic. OFFLINE: `scores=[4, 6, 8] revisions=2` for each (the scripted critic is topic-independent). LIVE: real scores, often `[6, 8]` or `[7, 9]`; a few seconds per topic.

## Step 5 · Cost and latency per run

**TODO 4** is `cost_latency_table` in the next cell.

In [ ]:
def cost_latency_table(run: dict) -> list[dict]:
    """One row per node: {"node", "calls", "seconds", "share"} (+ a "TOTAL" row with "tokens" and "usd")."""
    # TODO 4: where does the time (and money) go? One row per node from the run's run_log, each with its share of
    #         the total, then a TOTAL row with the run's tokens and estimated $ (run["usage"], estimate_cost).
    return []



for run in runs:
    table = cost_latency_table(run)
    print(f"\n{run['topic']}  (wall clock {run['seconds']}s)")
    print(f"{'node':<18}{'calls':>6}{'seconds':>10}{'share':>8}")
    for row in table:
        extra = f"   tokens={row['tokens']} ≈ ${row['usd']:.5f}" if row["node"] == "TOTAL" else ""
        print(f"{row['node']:<18}{row['calls']:>6}{row['seconds']:>10.4f}{row['share']:>8.0%}{extra}")

log = [{"topic": r["topic"], "scores": r["out"].get("scores", []), "revisions": r["out"].get("iteration"),
        "seconds": r["seconds"], "table": cost_latency_table(r)} for r in runs]
(OUT / "project07-runs.json").write_text(json.dumps(log, indent=2), encoding="utf-8")
print("\nsaved data/outputs/m07/project07-runs.json")

<details><summary>Hint for TODO 4 (open only if stuck)</summary>

`per = run_summary(run["out"].get("run_log", []))` gives `{node: {"calls", "seconds"}}`. Divide each node's seconds by the total for `share`. Tokens: sum `total_tokens` over `run["usage"].values()`; dollars: `estimate_cost(usage, MODEL_KEY) or 0` per model. Append a `TOTAL` row with `tokens` and `usd`.
</details>

**You should now see** a table per topic: `research_section` with 3 calls, `critique` 3, `revise` 2, and a `TOTAL` row. OFFLINE: milliseconds and `tokens=0 ≈ $0.00000`. LIVE: `write_draft`/`revise` dominate the seconds; ~6–12k tokens per topic ≈ $0.001–0.003 at Groq's paid prices (free on the free tier). Starter: only the headers until TODO 4.

**Traces:** with `LANGSMITH_API_KEY` and `LANGSMITH_TRACING=true` in `.env`, every run is also a LangSmith trace with the same per-node latency and tokens. Compare one trace with your table.

## Step 6 · Same app in Studio
Your graph and `apps/m07_content_pipeline/pipeline.py` must be **the same pipeline**. The app builds it with `mastery.graphs.content.build_content_graph` (the reference implementation of this project); this cell starts `langgraph dev`, reads the served graph's nodes and runs one topic through the HTTP API.

To click through it yourself (do this once): in a second Git Bash window run `cd apps/m07_content_pipeline && langgraph dev`, open the Studio link, run a topic and watch `critique` ⇄ `revise` light up.

In [ ]:
# The same app under `langgraph dev` (local, free). LIVE uses the key from .env, OFFLINE forces scripted models.
from langgraph_sdk import get_sync_client

from mastery.graphs.server import DevServer

app_nodes, server_result = None, None
if RUN_SERVER:
    server = DevServer(APP_DIR, env={} if LIVE else {"MASTERY_OFFLINE": "1"}, log_path=OUT / "project07-langgraph-dev.log")
    try:
        server.start()
    except Exception as exc:          # a port clash or slow start must not stop Restart & Run All
        print(f"⚠️ langgraph dev did not start ({type(exc).__name__}: {exc}). Log: data/outputs/m07/project07-langgraph-dev.log. Checks 6–7 are skipped.")
        server = None
    try:
        if server is None:
            raise RuntimeError("skip")
        print("Studio:", server.studio_url, "(open it now, the server stops at the end of this cell)")
        client = get_sync_client(url=server.url)
        graph_info = client.assistants.get_graph(client.assistants.search(graph_id="content_pipeline")[0]["assistant_id"])
        app_nodes = sorted(n["id"] for n in graph_info["nodes"] if not n["id"].startswith("__"))
        thread = client.threads.create()
        server_result = client.runs.wait(thread["thread_id"], "content_pipeline", input={"topic": TOPICS[0]})
        print("server scores:", server_result.get("scores"), "| notebook scores:", runs[0]["out"].get("scores"))
    except RuntimeError as exc:
        if str(exc) != "skip":
            raise
    finally:
        if server is not None:
            server.stop()
else:
    print("⏭️ Server step skipped (RUN_SERVER = False). Run `langgraph dev` in apps/m07_content_pipeline yourself.")

my_nodes = sorted(n for n in pipeline.get_graph().nodes if not n.startswith("__"))
print("notebook nodes:", my_nodes)
print("app nodes     :", app_nodes)

**You should now see** the Studio link, `server scores: [4, 6, 8] | notebook scores: [4, 6, 8]` (OFFLINE) and the same six node names twice. LIVE, scores can differ between the two runs (temperature 0 is not a guarantee); the node names can't.

## Step 7 · Acceptance checks

In [ ]:
# Acceptance checks.
from mastery.testing import ScriptedChatModel
from langchain_core.messages import AIMessage

ok = []
# 1. Deterministic termination: a critic that never approves must still stop after MAX_ITERATIONS revisions.
never_happy = ScriptedChatModel(responses=[AIMessage("", tool_calls=[{"name": "Critique", "id": "c",
                                                                     "args": {"score": 2, "issues": ["Rewrite it."]}}])],
                                repeat_last=True)
critic_model, saved_critic = never_happy.with_structured_output(Critique), critic_model
try:
    harsh = build_pipeline().invoke({"topic": TOPICS[0]}, {"recursion_limit": 40})
    stopped = harsh.get("iteration") == MAX_ITERATIONS and "final" in harsh
except Exception as exc:              # GraphRecursionError = the loop didn't stop on its own
    stopped, harsh = False, {"error": type(exc).__name__}
finally:
    critic_model = saved_critic
ok.append(check(f"terminates with a never-satisfied critic (revisions={harsh.get('iteration')})", stopped, "TODO 2/3"))

# 2. Parallel research collected by the reducer: one note per outline section, each with evidence.
ok.append(check("one research note per outline section, each with a source (Send + reducer)",
                all(len(r["out"].get("sections", [])) == len(r["out"]["outline"].sections) > 0
                    and all(s["sources"] for s in r["out"]["sections"]) for r in runs), "TODO 1"))

# 3. Quality you can check without a judge: every citation in the final post points at a source the research
#    returned, and there is at least one. (The critic's score is the model's opinion; this is evidence.)
CITATION = re.compile(r"\[source: ([^\]]+)\]")


def citations_grounded(out: dict) -> bool:
    # One tag may name several sources ("[source: a.md, b.md]", as the notes header lists them): check each one.
    cited = {name.strip() for tag in CITATION.findall(out.get("final", "")) for name in re.split(r"[,;]", tag) if name.strip()}
    found = {src for s in out.get("sections", []) for src in s["sources"]}
    return bool(cited) and cited <= found


def citation_problems(out: dict) -> str:
    """What a failing post got wrong, so you can tell a model slip from a bug in your graph."""
    cited = {n.strip() for tag in CITATION.findall(out.get("final", "")) for n in re.split(r"[,;]", tag) if n.strip()}
    found = {src for s in out.get("sections", []) for src in s["sources"]}
    return "no [source: ...] tags" if not cited else f"not in the research: {sorted(cited - found)}"


fake = {**runs[0]["out"], "final": runs[0]["out"].get("final", "") + " The moon is cheese. [source: made-up.md]"}
ok.append(check("every citation in every final post comes from the research (and a made-up one is caught)",
                all(citations_grounded(r["out"]) for r in runs) and not citations_grounded(fake),
                "; ".join(f"{r['topic']}: {citation_problems(r['out'])}" for r in runs if not citations_grounded(r["out"]))
                or "TODO 1/2"))
ok.append(check("at least one topic needed a revision (the loop is real)",
                any(r["out"].get("iteration", 0) >= 1 for r in runs) or LIVE, "TODO 2/3"))

# 4. Same app in the notebook and in Studio.
ok.append(check("notebook graph has the app's node names", None if app_nodes is None else my_nodes == app_nodes, "build_pipeline"))
ok.append(check("the server ran the same pipeline", None if server_result is None else
                (LIVE or server_result.get("scores") == runs[0]["out"].get("scores")), "TODO 1/2/3"))

# 5. Cost/latency logged per run.
ALWAYS = {"plan_outline", "research_section", "write_draft", "critique", "finalize"}   # "revise" only if needed
ok.append(check("cost/latency table per run (every node that ran + TOTAL)",
                all({row["node"] for row in cost_latency_table(r)} >= set(run_summary(r["out"].get("run_log", []))) | ALWAYS | {"TOTAL"}
                    for r in runs), "TODO 4"))

done = [o for o in ok if o is not None]
print(f"\n{sum(done)}/{len(done)} acceptance checks passed" + (f", {ok.count(None)} skipped" if None in ok else ""))

**You should now see** `7/7 acceptance checks passed` (solution; with `RUN_SERVER = False` or a server that didn't start: `5/5 acceptance checks passed, 2 skipped`, never a silent ✅). Starter: ❌ next to the TODO to fix. Check 1 is the important one: it swaps in a critic that is **never** satisfied and proves your counter ends the loop (without it, LangGraph would stop at `recursion_limit` with an error instead).

## Step 8 · Stretch: the Functional API version

In [ ]:
# Stretch (TODO 5, optional): the same pipeline with the Functional API (offline models, so the scores are comparable).
from langgraph.func import entrypoint, task

if RUN_STRETCH:
    off = offline_models()
    f_plan, f_critic, f_writer = (off["planner"].with_structured_output(Outline),
                                  off["critic"].with_structured_output(Critique), off["writer"])

    @task
    def f_research(title: str, question: str) -> dict:
        notes, sources = researcher(question)
        return SectionNote(title=title, notes=notes, sources=sources)

    # TODO 5: an @entrypoint() functional_pipeline(topic) -> {"scores", "iteration", "final"} that does what the graph
    #         does: plan, research every section in parallel (f_research futures), write, then critique/revise until
    #         THRESHOLD or MAX_ITERATIONS. Lesson 07.05: put every model call and other I/O inside a @task.
    print("⏭️ TODO 5 (stretch) not written yet.")
else:
    print("⏭️ Stretch skipped (RUN_STRETCH = False).")

**You should now see** (solution) `functional scores=[4, 6, 8]` for each topic; in the starter, the ⏭️ line until you write TODO 5. The solution puts every model call in a `@task` (Lesson 07.05's rule), so a resumed run replays saved results instead of calling the model again. It is the same behaviour in one ~35-line function (tasks included) instead of six nodes plus wiring (~70 lines). The functional version is a plain `while` loop, shorter and easy to read top to bottom; the graph version is longer but you can **see** it (Step 3), stream node by node, and fork it in Studio. Write which one you'd maintain, and why.

## Findings (write yours)

| | Topic 1 | Topic 2 | Topic 3 |
|---|---|---|---|
| scores per iteration | | | |
| revisions | | | |
| seconds (wall clock) | | | |
| tokens / $ | | | |

- Did the critic's score really improve the draft, or just the score? Read one revision.
- Where does the time go? Which node would you speed up first (smaller model, fewer sections, parallel critique)?
- Graph or Functional API for this pipeline? Log the decision in [`docs/decision-log.md`](../../../docs/decision-log.md).

## ✅ Before you commit

- [ ] Restart & Run All passes (LIVE and without a key).
- [ ] All TODOs done; 7/7 acceptance checks; findings written.
- [ ] I ran the app in Studio myself once.
- [ ] Outputs cleared (`jupyter nbconvert --clear-output --inplace <notebook>`); no keys in outputs; `data/outputs/` not committed.

---
## ➡️ What's next

**Next:** [Module 08 · LangGraph Production Features](../../../modules/08-langgraph-production-features/README.md): persistence, durable execution, interrupts, time travel, subgraphs, streaming and stores. Start with its README's **Before you start this module** block (pull, install, verify). Its project, a research-and-write agent, adds what this pipeline lacks: a SQLite checkpointer that survives crashes, a human approval step, and forking from the plan.